# Angle session

The A1335's registers, and whether there is a magnet.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

In [2]:
from coaxial import Coaxial63100

device = Coaxial63100(port=PORT, simulated_device=SIMULATED).open()
print(device)

<Coaxial63100 Simulated SIMULATED>


The A1335 sits on SPI4 behind AFE_ON. The poll loop reads one register into shared memory; `state()` reads that record and touches no SPI.

In [3]:
device.afe.enable()
angle = device.angle
st = angle.state()
print({k: st[k] for k in ('loop', 'updates', 'register_name', 'value', 'degrees', 'crc')})
print(angle.clock())
print(angle.poll_register())

{'loop': 'running', 'updates': 37, 'register_name': 'ANG', 'value': 20482, 'degrees': 0.17578125, 'crc': 0}
{'kernel_hz': 118750000, 'bitrate_hz': 1855468}
{'register': 32, 'register_name': 'ANG'}


Registers, from the reference implementation rather than the datasheet in this tree: ANG 0x20, STA 0x22, ERR 0x24, XERR 0x26, TSEN 0x28, FIELD 0x2A. A read is two frames; the CRC is reported, not checked. Direct reads need the loop held.

In [4]:
from coaxial.angle import degrees, kelvin, gauss

with angle.configuring():
    for reg in (0x20, 0x22, 0x24, 0x26, 0x28, 0x2A):
        got = angle.read(reg)
        print('%-5s 0x%04X  crc %d' % (got['register_name'], got['value'], got['crc']))
    ang = angle.read(0x20)['value']
    tsen = angle.read(0x28)['value']
    field = angle.read(0x2A)['value']
print('angle  %.2f deg' % degrees(ang))
print('die    %.1f K' % kelvin(tsen))
print('field  %.0f G' % gauss(field))

ANG   0x5003  crc 0
STA   0x8000  crc 0
ERR   0x8000  crc 0
XERR  0x8000  crc 0
TSEN  0xF940  crc 0
FIELD 0xE17C  crc 0
angle  0.35 deg
die    296.0 K
field  380 G


FIELD reads about 2 G with no magnet on the real board; 300 to 1000 G is the recommended range. The stand-in reports a magnet in place.

In [5]:
import time

turning = []
for _ in range(6):
    st = angle.state()
    turning.append((time.monotonic(), st['degrees'], st['updates']))
    print('%8.2f deg  updates %d' % (st['degrees'], st['updates']))
    time.sleep(0.25)
device.close()

    0.44 deg  updates 74


    8.00 deg  updates 111


   15.47 deg  updates 148


   23.03 deg  updates 185


   30.59 deg  updates 222


   38.06 deg  updates 259


## Conclusions

In [6]:
span = turning[-1][0] - turning[0][0]
moved = turning[-1][1] - turning[0][1]
print('shaft            %.2f deg over %.2f s' % (moved, span))
print('updates          %d in that window = %.0f /s'
      % (turning[-1][2] - turning[0][2], (turning[-1][2] - turning[0][2]) / span))
print('ANG  0x%04X      low 12 bits x 360/4096 = %.2f deg' % (ang, degrees(ang)))
print('TSEN 0x%04X      eighths of a kelvin    = %.1f K = %.1f C'
      % (tsen, kelvin(tsen), kelvin(tsen) - 273.15))
print('FIELD 0x%04X     %.0f gauss' % (field, gauss(field)))
print('CRC              reported, not checked')

shaft            37.62 deg over 1.25 s
updates          185 in that window = 148 /s
ANG  0x5004      low 12 bits x 360/4096 = 0.35 deg
TSEN 0xF940      eighths of a kelvin    = 296.0 K = 22.9 C
FIELD 0xE17C     380 gauss
CRC              reported, not checked


Every read is two frames: the address arrives on MOSI bits 17..12 while MISO has already shifted out bits 19..16, so the answer cannot be to the frame carrying the address - asking TSEN, FIELD, TSEN in turn returned the previous register's value every time. The first frame posts the address, the second clocks the answer out.

The CRC is reported and not checked: the datasheet in this tree gives the field's width and not its polynomial, and checking against a guessed one would reject good readings. The register map came from a reference implementation rather than that datasheet, which is why the polled register is settable without a rebuild.

FIELD says whether there is a magnet: the real board reads about 2 gauss with none, and 300 to 1000 is the recommended range. TSEN is the part's own die, not the board - it quantises at 0.125 K and is reset every time AFE_ON breaks; measured 2026-08-28 it fell 1.88 K during a run that warmed the board, which is why the NTC is the thermal observer's reference and this is not.